# Hour 05: Matrix Multiplication and Neural Network Layers

In the previous hour, we calculated the output of a single neuron. But real neural networks have **layers** containing multiple neurons. To process a batch of data simultaneously through a layer with multiple neurons, we need **Matrix Multiplication**.

In [3]:
import numpy as np

## 1. The Shape Rule for Matrix Multiplication

To multiply two matrices using the `@` operator (or `np.matmul`), the **inner dimensions must match**.

If Matrix $A$ has shape $(m, n)$ and Matrix $B$ has shape $(n, p)$:
*   The inner dimension is $n$ (they match!).
*   The resulting Matrix $C$ will have the shape of the outer dimensions: $(m, p)$.

In [4]:
A = np.array([
    [1, 2],
    [3, 4],
    [5, 6]
]) # Shape: (3, 2)

B = np.array([
    [7, 8, 9],
    [10, 11, 12]
]) # Shape: (2, 3)

C = A @ B
print(f"Shape of A: {A.shape}")
print(f"Shape of B: {B.shape}")
print(f"Shape of C (A @ B): {C.shape}")
print("\nMatrix C:\n", C)

Shape of A: (3, 2)
Shape of B: (2, 3)
Shape of C (A @ B): (3, 3)

Matrix C:
 [[ 27  30  33]
 [ 61  68  75]
 [ 95 106 117]]


## 2. Data Pipeline Mechanics: Shuffling and Stacking Batches

Before feeding batches of data into a dense layer matrix multiplication, you must prepare the data. Two essential pipeline tasks are shuffling datasets (to avoid training bias) and combining feature vectors.

### A. Dataset Shuffling (`np.random.permutation`)
To shuffle a feature matrix $X$ and its corresponding targets $y$ without losing the relationship between rows, use a randomized index permutation.

In [16]:
n= 5
np.random.seed(42)
shuffle_indices = np.random.permutation(n)
print(shuffle_indices)

shuffle_indices = np.random.permutation(n)
print(shuffle_indices)

[1 4 2 0 3]
[3 1 2 0 4]


In [18]:
n= 5
np.random.seed(43)
shuffle_indices = np.random.permutation(n)
print(shuffle_indices)

shuffle_indices = np.random.permutation(n)
print(shuffle_indices)

[3 2 1 0 4]
[3 4 1 0 2]


In [19]:
n= 5
np.random.seed(42)
shuffle_indices = np.random.permutation(n)
print(shuffle_indices)

shuffle_indices = np.random.permutation(n)
print(shuffle_indices)

[1 4 2 0 3]
[3 1 2 0 4]


In [5]:
# Create a dummy dataset: 4 samples, 2 features
X_data = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
y_data = np.array([0, 1, 0, 1])

# Generate a shuffled array of indices
np.random.seed(42)
shuffled_indices = np.random.permutation(len(X_data))
print("Shuffled Index Order:", shuffled_indices)

# Apply indices to both arrays to keep features aligned with targets
X_shuffled = X_data[shuffled_indices]
y_shuffled = y_data[shuffled_indices]

print("\nShuffled X:\n", X_shuffled)

Shuffled Index Order: [1 3 0 2]

Shuffled X:
 [[3 4]
 [7 8]
 [1 2]
 [5 6]]


### B. Combining Data (np.vstack and np.hstack)
When combining data arrays, you can stack them vertically (adding samples) or horizontally (adding features).

- ```np.vstack()```: Stacks arrays along the row axis (vertical).

- ```np.hstack()```: Stacks arrays along the column axis (horizontal).

In [15]:
batch_1 = np.array([[1, 2], [3, 4]])
batch_2 = np.array([[5, 6], [7, 8]])

# Stack vertically to create a larger batch
combined_batches = np.vstack((batch_1, batch_2))
print("Vertically Stacked (New Rows):\n", combined_batches)

combined_batches = np.hstack((batch_1, batch_2))
print("Horizontally Stacked (New Cols):\n", combined_batches)

# Stack horizontally to append new features to existing samples
# extra_features = np.array([[10], [20]])
# extended_batch = np.hstack((batch_1, extra_features))
# print("\nHorizontally Stacked (New Column):\n", extended_batch)

Vertically Stacked (New Rows):
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]
Horizontally Stacked (New Cols):
 [[1 2 5 6]
 [3 4 7 8]]


## 3. Explicit Dimension Control: `np.expand_dims` and `np.squeeze`

While `.reshape()` is highly flexible, deep learning workflows frequently require you to specifically add or remove "singleton" dimensions (axes of size 1). This is common when converting a 1D array of targets `(N,)` into a 2D column vector `(N, 1)` to calculate loss, or when stripping out batch dimensions for single-sample inference.

### A. Adding Dimensions (`np.expand_dims`)
Use `np.expand_dims()` to inject a new axis of size 1 at a specific position. This is a **View** operation.

In [24]:
# A 1D array representing 3 labels (e.g., target values)
y = np.array([5, 10, 15])
print(f"Original Shape: {y.shape}: {y}")  # Output: (3,)

# Expand to a 2D column vector (axis=1)
y_col = np.expand_dims(y, axis=1)
print(f"Column Vector Shape:: {y_col.shape}:\n{y_col}")  # Output: (3, 1)

# Expand to a 2D row vector (axis=0)
y_row = np.expand_dims(y, axis=0)
print(f"Row Vector Shape:: {y_row.shape}:\n{y_row}")  # Output: (1, 3)

Original Shape: (3,): [ 5 10 15]
Column Vector Shape:: (3, 1):
[[ 5]
 [10]
 [15]]
Row Vector Shape:: (1, 3):
[[ 5 10 15]]


### B. Removing Dimensions (```np.squeeze```)
Use ```np.squeeze()``` to drop all dimensions that have a size of exactly 1. It collapses empty axes without affecting the actual data. This is also a View operation.

In [29]:
# A model output tensor for a single sample batch: 1 batch, 1 sample, 3 class logits
model_output = np.array([[[2.5, 0.1, 0.8]]])
print("Complex Shape:", model_output.shape)  # Output: (1, 1, 3)

# Squeeze out the empty singleton dimensions
flat_output = np.squeeze(model_output)
print("Squeezed Shape1:", flat_output.shape)  # Output: (3,)
print("Cleaned Vector1:", flat_output)

flat_output = np.squeeze(model_output, axis=0)
print("Squeezed Shape2:", flat_output.shape)  # Output: (1, 3,)
print("Cleaned Vector2:", flat_output)

flat_output = np.squeeze(flat_output, axis=0)
print("Squeezed Shape3:", flat_output.shape)  # Output: (3,)
print("Cleaned Vector3:", flat_output)


# flat_output = np.squeeze(model_output, axis=2)
# print("Squeezed Shape4:", flat_output.shape)  # Output: (3,)
# print("Cleaned Vector4:", flat_output)

Complex Shape: (1, 1, 3)
Squeezed Shape1: (3,)
Cleaned Vector1: [2.5 0.1 0.8]
Squeezed Shape2: (1, 3)
Cleaned Vector2: [[2.5 0.1 0.8]]
Squeezed Shape3: (3,)
Cleaned Vector3: [2.5 0.1 0.8]


## 4. Building a Full Dense Layer

In a Dense (Fully Connected) layer, the formula is still $$ Z = X @ W + b $$, but the dimensions change:
*   **Inputs ($X$)**: Shape $(m, n)$ where $m$ is the number of samples, and $n$ is the number of input features.
*   **Weights ($W$)**: Shape $(n, p)$ where $p$ is the number of neurons in the layer.
*   **Bias ($b$)**: Shape $(p,)$ — one bias per neuron.
*   **Outputs ($Z$)**: Shape $(m, p)$ — the output of each neuron for every single sample.

In [7]:
X = np.array([
    [0.1, 0.5], # Sample 1
    [0.2, 0.3], # Sample 2
    [0.8, 0.1]  # Sample 3
]) # Batch of 3 samples, 2 features. Shape: (3, 2)

# A layer with 4 neurons
W = np.array([
    [0.2, 0.8, -0.5, 1.0],
    [0.5, -0.9, 0.2, -0.5]
]) # Weights shape: (2, 4) - 2 input features matching X, 4 neurons.

b = np.array([0.1, 0.0, -0.1, 0.5]) # Bias shape: (4,) - one bias per neuron

Z = (X @ W) + b # Broadcasting automatically adds the bias values to all 3 samples!
print("Layer Output (Z):\n", Z)
print("\nOutput Shape:", Z.shape) # Expected: (3, 4)

Layer Output (Z):
 [[ 0.37 -0.37 -0.05  0.35]
 [ 0.29 -0.11 -0.14  0.55]
 [ 0.31  0.55 -0.48  1.25]]

Output Shape: (3, 4)


## 🛠 Mini-Project: Forward Pass Simulator

Imagine you are feeding a batch of 5 images into a neural network layer. 
Each image has been flattened into an array of 10 pixel values. 
The layer has 3 neurons.

**Your Task:** 
1. Calculate the layer's output (`Z_layer`).
2. Print the shape of `Z_layer` to verify it is `(5, 3)`.

In [8]:
# Generating random data for simulation
np.random.seed(42) 
X_images = np.random.rand(5, 10)  # 5 images, 10 pixels each
W_neurons = np.random.rand(10, 3) # Weights for 3 neurons
b_neurons = np.random.rand(3)     # Biases for 3 neurons

# Write your code here:
Z_layer = ...

print("Layer Output Shape:", Z_layer.shape)
# print("Layer Outputs:\n", Z_layer)

AttributeError: 'ellipsis' object has no attribute 'shape'